In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import datetime

#Colour blind friendly colour palette to cycle through the different data sets
colourCycle = ['#377eb8', '#ff7f00', '#4daf4a',
              '#f781bf', '#a65628', '#984ea3',
              '#999999', "#e70004", "#8b8b02"]

#Dictionary to label all the channels appropriately, assign them a unique colour and marker. 
Detectors = {'Ch_4': ["LSC Left",colourCycle[0],"o"],
             'Ch_5': ["LSC Right",colourCycle[1],"V"],
             'Ch_8': ["NaI 1",colourCycle[2],"^"],
             'Ch_10': ["NaI 1",colourCycle[3],"<"],
             'Ch_12': ["NaI 1",colourCycle[4],"P"],
             'Ch_14': ["NaI 1",colourCycle[5],"S"]
             }

In [14]:
class ProfileHistData:
    # This class stores all of the data for a given detector.
    
    def __init__(self,ch):
        self.ch = ch
        self.data = []
        self.chLabel = Detectors[f'Ch_{self.ch}'][0]
        
        self.scaleFactors = []
        self.scaleFactorsErr = []
        self.dates = []
        
        self.colours = Detectors[f'Ch_{self.ch}'][1]
        self.marker = Detectors[f'Ch_{self.ch}'][2]
    
    def addFitResults(self,fitResults):
        self.data.append(fitResults)
        self.scaleFactors.append(fitResults.scaleFactor)
        self.scaleFactorsErr.append(fitResults.scaleFactorErr)
        self.dates.append(fitResults.date)
        
        
class fitData:
    #This class is designed to be a container to hold the individual information about each channels fits. 
    
    def __init__(self,ch,scaleFactor,scaleFactorErr,date):
        self.ch = ch
        self.scaleFactor = scaleFactor
        self.scaleFactorErr = scaleFactorErr
        self.date = date
        
    

In [15]:
def ReadInProfileData(filepath):
    #Currently only reading in all the 
    gammaEnergy = 662 #keV
    chResults,totalData = [],[] #chResults is the list of all fitData. totalData is the histogram of ProfileHistData
    Channels = [4,5,8,10,12,14]
    
    for ch in Channels: #initialize each ProfileHistData instance for each 
        totalData.append(ProfileHistData(ch))
        
    for file in filepath:
        
        with open(file) as f:
            next(f) #skips the header of the file. 
            for line in f:
                data = line.split(",")
                time = datetime.datetime(int(data[0].split("_")[0]),int(data[0].split("_")[1]),int(data[0].split("_")[2]))
                channels = [int(data[1]),int(data[2])]
                p0 = float(data[8])
                p0Err = float(data[9])
                p1 = float(data[10])
                p1Err = float(data[11])
                
                
                #note: the x axis is always the liquid scintillator, and corresponds to the first channel listed. 
                #      The second channel is always the NaI detectors, and corresponds to the second channel listed. 
                xInt = -p0/p1
                yInt = p0
                
                xIntErr = xInt*np.sqrt((p0Err/p0)**2 + (p1Err/p1)**2)
                yIntErr = p0Err
        
                if channels[0] == 4 and channels[1] == 5:
                    continue
                else:
                    #(self,ch,scaleFactor,scaleFactorErr,date):
                    chResults.append(fitData(channels[0],xInt,xIntErr,time)) #Save all the data in the LSC to a fitData object.
                    chResults.append(fitData(channels[1],yInt,yIntErr,time)) #Save all the data in the NaI to a fitData object.
                    
                    for totD in totalData: 
                        if chResults[-1].ch == totD.ch:
                            totD.addFitResults(chResults[-1])
                        if chResults[-2].ch == totD.ch:
                            totD.addFitResults(chResults[-2])
                            
    return chResults, totalData

    
    
    

In [21]:
rootFilePath = Path('/home/nick/PhD/KDK+/Daily_LSC_Calibration_testing/') #This is the filepath to the overall directory that holds all the calibration data. 
ResultsFilePaths = rootFilePath.rglob('*_Profile_hist_fit_results.txt')

chResults, totalData = ReadInProfileData(ResultsFilePaths)

In [ ]:
fig,ax = plt.subplots(2,1,figsize = (20,10))

